In [10]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolTransforms
import subprocess
import os
import numpy as np
import tempfile
import shutil

def calculate_quinazoline_docking(smiles, pdb_id="4XCU", pdb_path="4xcu.pdb"):
    """
    计算喹唑啉衍生物与FGFR4(PDB:4XCU)的对接分数
    参数:
        smiles (str): 含*标记的喹唑啉骨架SMILES（如Clc1nc2ccccc2cc1*）
        pdb_id (str): 蛋白质PDB ID
        pdb_path (str): 本地PDB文件路径
    返回:
        float: AutoDock Vina对接分数
    """
    # 1. 验证输入SMILES格式
    # if '*' not in smiles:
    #     raise ValueError("SMILES字符串必须包含*标记修饰位点")
    
    # 2. 创建临时工作目录
    # temp_dir = tempfile.mkdtemp()
    # os.chdir(temp_dir)

    # 3. SMILES转3D结构
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        raise ValueError("无效SMILES字符串")
    
    # 添加氢原子并生成3D构象
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol)  # 力场优化
    
    # 4. 保存为SDF格式（AutoDock兼容格式）
    ligand_sdf = "./ligand.sdf"
    writer = Chem.SDWriter(ligand_sdf)
    writer.write(mol)
    writer.close()
        
    
    try:
        
        # 5. 准备受体文件（若不存在则下载）
        # if not os.path.exists(pdb_path):
        #     subprocess.run(["wget", f"http://files.rcsb.org/download/{pdb_id}.pdb"], check=True)
        
        # 6. 使用AutoDock Vina进行对接
        # 参考BLU9931共晶结构口袋坐标 [[5]][[10]]
        cmd = [
            "vina",
            "--receptor", pdb_path,
            "--ligand", ligand_sdf,
            "--center_x", "10.2",
            "--center_y", "35.7",
            "--center_z", "22.4",
            "--size_x", "20",
            "--size_y", "20",
            "--size_z", "20",
            "--score_only"
        ]
        print(cmd)
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
       
        
        # 7. 解析结果
        score = None
        for line in result.stdout.split('\n'):
            if line.startswith('Affinity:'):
                score = float(line.split()[1])
                break
        
        if score is None:
            raise RuntimeError("未找到有效对接分数")
            
        return score
    
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"对接执行失败: {e.stderr}") from e
    
    finally:
        # 8. 清理临时文件
        os.chdir("..")
        shutil.rmtree(temp_dir)

# 示例用法
if __name__ == "__main__":
    # 喹唑啉骨架（4号位带修饰位点*）
    example_smiles = "Clc1nc2ccccc2cc1(CC)"  # [[2]]
    
    # 设置受体文件路径
    receptor_path = "4XCU.pdb"
    
    # 计算对接分数
    try:
        score = calculate_quinazoline_docking(example_smiles, pdb_path=receptor_path)
        print(f"对接分数: {score} kcal/mol")
    except Exception as e:
        print(f"计算失败: {str(e)}")

计算失败: File error: Bad output file ./ligand.sdf


In [12]:
mol = Chem.MolFromSmiles("Clc1nc2ccccc2cc1(CC)")
if not mol:
    raise ValueError("无效SMILES字符串")

# 添加氢原子并生成3D构象
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.UFFOptimizeMolecule(mol)  # 力场优化

# 4. 保存为SDF格式（AutoDock兼容格式）
ligand_sdf = "./ligand.sdf"
writer = Chem.SDWriter(ligand_sdf)
writer.write(mol)
writer.close()

OSError: File error: Bad output file ./ligand.sdf